# 02 - Model training & evaluation

Train the LightGBM demand model with a time-based holdout, compare it with a
seasonal-naive baseline, and inspect feature importance. Set `use_mlflow=True`
to log the run (then `mlflow ui`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

from src.forecasting.data import load_sales
from src.forecasting.predict import DemandForecaster
from src.forecasting.train import train

sales = load_sales(ROOT / "data/raw/sample_sales.csv")
result = train(sales, model_path=ROOT / "models/demand_model.joblib", holdout_days=56, tune=True, use_mlflow=False)
pd.DataFrame({"LightGBM": result.metrics, "Seasonal naive": result.baseline_metrics}).round(3)

## Feature importance

In [ ]:
model = result.bundle["model"]
pd.Series(model.feature_importances_, index=result.bundle["features"]).sort_values(ascending=False)

## Recursive 28-day forecast

In [ ]:
forecaster = DemandForecaster.load(result.model_path)
forecaster.forecast("SKU-101", horizon_days=28).head(10)